# Set Up pwd and auto updates

In [ ]:
import os
from pathlib import Path
import subprocess

# Get the top-level directory of the current git repo
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()
)

os.chdir(PROJECT_ROOT)

# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2




# Ensure external data exists

In [ ]:
!python collect_external_data/expected_counts.py
!python collect_external_data/road_geom.py

# Sim related imports

In [ ]:
from season.persons import SeasonPerson
from season.configs import ScheduleSpecs, SeasonConfig, DayParams, make_season_config, PopulationParams
from season.season_orchestrator import SeasonOrchestrator
from traffic.model.tolling import TollConfig, VolumeSignal, FlowSignal, PiecewiseLinearTransform, StepTransform, PITransform

import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.stats import norm, lognorm, skewnorm, truncnorm, uniform

# Mesa model components
import mesa
from mesa import model
from mesa import agent
from traffic.model.traffic_model import TrafficModel
from traffic.agents.vehicle_agent import VehicleAgent
from traffic.agents.road_segment_agent import RoadSegmentAgent

import traffic.utils.unit_conversion_utils as uc
import traffic.utils.analysis_utils as au
import traffic.utils.animation_utils as anim

# Visualization and display (Jupyter-specific)
from IPython.display import display, HTML
pd.set_option("display.max_rows", None)  

import subprocess

# Define the config

In [ ]:

config = make_season_config(
    season_id='exp_data_01',
    run_description='One day test run',
    seed=124,
    n_days=1,
    max_steps=10000,
    max_persons=2000,
    collect_every_n=5,
    batch_run=False, # must be false if you want to do animations

    start_hr=8,
    bus_capacity=30,
    road_path='data/roads/hw210_sl_and_curvs.parquet', # files should be saved here by default 
    ecs_path='data/vehicle_counts/expected_counts_seconds.csv', # files should be saved here by default 
    
    # Tolling config - use TollConfig.static() for fixed tolls
    toll=TollConfig.static(car=10.0),
    bus_user_fee=0.0,
   
    traffic_percentile_schedule=ScheduleSpecs(mode='static', value=50),  
    bus_interval_schedule=ScheduleSpecs(mode='static', value=15), # static bus interval of 15 minutes,
    crashes_schedule=ScheduleSpecs(mode='static', value=0), 
    population_params = PopulationParams(
                                        population_size=2000,
                                        prior_bus=40.0,

),
    canyon_closures_schedule=None,
)

config

# Run single day

a simple single day run with appropriate analysis

In [ ]:
# Example usage of SeasonOrchestrator with example_config
orchestrator = SeasonOrchestrator(season_config=config)
tm = orchestrator.run_day()

## Analysis

In [ ]:
finished_agents = au.finished_agents_summary_df(tm, plots=True)
len(finished_agents)

In [ ]:
# process the finished_agents data 
vehicles_full = au.vehicle_agent_data_time_series(tm, plots=True)
model_ts = tm.datacollector.get_model_vars_dataframe()


au.plot_speed_delta(vehicles_full, model_ts)

## Annimations

In [ ]:
# run the animation
# looking at one vehicle - make sure issue_car_id is an anctual vehicle
issue_car_id =  None
issue_step = 0

road_gdf = gpd.read_parquet('data/roads/hw210_sl_and_curvs.parquet') # this is needed for animations

In [ ]:
anim.animate_traffic(vehicles_full, road_gdf, interval=100, step_skip=2, watch=issue_car_id, zoom=20)

# a much larger version of the previous, also shows a updating graph
#anim.animate_traffic_with_speed_delta_highlight(vehicles_full, road_gdf, model_ts, interval=100, step_skip=1, watch=None, zoom=20)

In [ ]:
anim.animate_relative_distance(vehicle_df=vehicles_full, agent_id=4454, distance_behind=100, color_by='driving_action')

# Peram sweep 

use this to test collect data for many single days

In [ ]:
from copy import deepcopy
from itertools import product


def make_experiment_configs(
    base_config,
    n_samples,
    traffic_percentiles,
    bus_schedules,
    bus_priors,
    car_tolls,
    base_seed=1,
):
    """
    Create a list of configs by varying traffic_percentile, bus_prior, car_toll,
    and repeating each combination n_samples times with unique seeds.

    Total configs = n_samples * len(traffic_percentiles) * len(bus_priors) * len(car_tolls)
    """
    configs = []
    seed_counter = base_seed

    for tp, bus_prior, car_toll, bus_int in product(traffic_percentiles, bus_priors, car_tolls, bus_schedules):
        for _ in range(n_samples):
            cfg = deepcopy(base_config)

            # unique seed for this config
            seed = seed_counter
            seed_counter += 1

            # global/season seed
            cfg.seed = seed

            # bus prior + toll
            cfg.population_params.prior_bus = bus_prior
            cfg.toll_config = TollConfig.static(car=car_toll)

            # day-level params
            day_cfg = cfg.day_params[0]
            day_cfg.traffic_percentile = tp
            day_cfg.bus_interval = bus_int
            day_cfg.day_seed = seed

            configs.append(cfg)
    
    print(len(configs), "experiment configurations created.")

    return configs

def append_row_to_csv(row, csv_path):
    """
    row: dict
    csv_path: 'data/traffic_experiment/experiment_results.csv'
    """
    # make sure folder exists
    folder = os.path.dirname(csv_path)
    if folder:
        os.makedirs(folder, exist_ok=True)

    df = pd.DataFrame([row])

    if os.path.exists(csv_path):
        # append without header
        df.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        # first write with header
        df.to_csv(csv_path, index=False)

In [ ]:
bus_schedules = [5, 10 ,30]  # minutes
traffic_percentiles = [50, 70, 80, 90, 95]
bus_priors = [30,40,50]          # minutes
car_tolls = [0, 5, 10, 20]#[0, 5, 10, 15, 20]     # dollars

experiment_configs = make_experiment_configs(
    base_config=config,
    n_samples=2,
    traffic_percentiles=traffic_percentiles,
    bus_schedules=bus_schedules,
    bus_priors=bus_priors,
    car_tolls=car_tolls,
    base_seed=1,
)



In [ ]:
# lol runs can be very long

caff = subprocess.Popen(["caffeinate", "-di"])

csv_path = "data/traffic_experiment/single_day_experiment_results.csv"

for cfg in experiment_configs:
    orchestrator = SeasonOrchestrator(cfg)
    row = orchestrator.run_day_temp()

    print(row)
    append_row_to_csv(row, csv_path)


caff.terminate() 